## nb_02_landing_to_bronze
Download files from kaggle to landing zone and ingest them into bronze schema
Since we are just downloading and loading into bronze schema a single notebook will do
as they handle the same datasets.

In [1]:
                                                                                                                                                                                    # Install kagglehub module
!pip install kagglehub --quiet

StatementMeta(, bcdf9bce-8279-49e7-b1e0-90803ba6c6f0, 3, Finished, Available, Finished, False)

### Define functions to download from kaggle to landing

In [2]:
from typing import Dict, List, Optional

import os
import kagglehub
import pandas as pd
from pyspark.sql import Row

WORKSPACE: str = "JDE10"
LAKEHOUSE: str = "NHL_db"

ABFSS_BASE_PATH: str = f"abfss://{WORKSPACE}@onelake.dfs.fabric.microsoft.com"
LANDING_PATH: str = f"{ABFSS_BASE_PATH}/{LAKEHOUSE}.Lakehouse/Files/landing"

def get_active_configs(spark) -> List[Row]:
    """Fetch active dataset configs from dbo.config_datasets.

    Raises ValueError if no active configs are found — there's nothing
    meaningful to do downstream if this table is empty or misconfigured.
    """
    query = """
        SELECT table_name, source_file, kaggle_dataset
        FROM dbo.config_datasets
        WHERE is_active = true;
    """
    configs = spark.sql(query).collect()
    if not configs:
        raise ValueError("No active rows found in dbo.config_datasets — nothing to download.")
    return configs


def group_by_kaggle_dataset(configs: List[Row]) -> Dict[str, List[Row]]:
    """Group config rows by kaggle_dataset so each dataset is downloaded only once,
    even if multiple source_files come from the same dataset."""
    datasets_needed: Dict[str, List[Row]] = {}
    for row in configs:
        datasets_needed.setdefault(row.kaggle_dataset, []).append(row)
    return datasets_needed


def download_kaggle_dataset(kaggle_dataset: str) -> str:
    """Download a Kaggle dataset via kagglehub.

    Raises RuntimeError with a clear message on auth/network failure,
    rather than letting kagglehub's raw exception surface.
    """
    try:
        download_path = kagglehub.dataset_download(kaggle_dataset)
    except Exception as e:
        raise RuntimeError(
            f"Failed to download Kaggle dataset '{kaggle_dataset}'. "
            f"Check that Kaggle credentials are configured for this environment. "
            f"Original error: {e}"
        ) from e

    if not os.path.isdir(download_path):
        raise RuntimeError(
            f"kagglehub reported success but download path does not exist: {download_path}"
        )
    return download_path


def build_file_lookup(download_path: str) -> Dict[str, str]:
    """Build a filename -> local path lookup for every file in the downloaded dataset."""
    available_files: Dict[str, str] = {}
    for root, _dirs, files in os.walk(download_path):
        for file_name in files:
            available_files[file_name] = os.path.join(root, file_name)

    if not available_files:
        print(f"⚠️ No files found under {download_path}")
    return available_files


def save_file_to_landing(
    local_file: str,
    source_file: str,
    table_name: str,
    landing_path: str = LANDING_PATH,
) -> bool:
    """Read a local CSV and write it to the landing zone.

    Returns True on success, False on any handled failure (bad CSV,
    empty file, write error) so the caller can track and report it
    without crashing the whole run over one bad file.
    """
    try:
        df = pd.read_csv(local_file)
    except (pd.errors.EmptyDataError, pd.errors.ParserError, OSError) as e:
        print(f"⚠️ Could not read '{local_file}' for table '{table_name}': {e} — skipping.")
        return False

    if df.empty:
        print(f"⚠️ '{source_file}' is empty (0 rows) for table '{table_name}' — skipping.")
        return False

    lakehouse_file_path = f"{landing_path}/{source_file}"
    try:
        df.to_csv(lakehouse_file_path, index=False)
    except OSError as e:
        print(f"⚠️ Failed to write '{source_file}' to landing zone: {e} — skipping.")
        return False

    print(f"✅ Saved {source_file} → landing/{source_file} (for table '{table_name}')")
    return True

StatementMeta(, bcdf9bce-8279-49e7-b1e0-90803ba6c6f0, 4, Finished, Available, Finished, False)

### Run: Kaggle → landing

In [3]:
configs = get_active_configs(spark)
datasets_needed = group_by_kaggle_dataset(configs)

failed_landings: List[str] = []

for kaggle_dataset, rows in datasets_needed.items():
    print(f"⬇️ Downloading dataset: {kaggle_dataset}")
    try:
        download_path = download_kaggle_dataset(kaggle_dataset)
    except RuntimeError as e:
        print(f"❌ {e}")
        failed_landings.extend(row.table_name for row in rows)
        continue
    print(f"Dataset downloaded to: {download_path}")

    available_files = build_file_lookup(download_path)

    for row in rows:
        source_file = row.source_file
        table_name = row.table_name

        if source_file not in available_files:
            print(f"⚠️ {source_file} not found for table '{table_name}' — skipping.")
            failed_landings.append(table_name)
            continue

        local_file = available_files[source_file]
        if not save_file_to_landing(local_file, source_file, table_name):
            failed_landings.append(table_name)

if failed_landings:
    print(f"\n⚠️ {len(failed_landings)} table(s) failed to land: {failed_landings}")
else:
    print("\n🏁 Landing zone load complete.")

# Fail loudly only if EVERYTHING failed — a partial landing still lets
# nb_01's own bronze step (and downstream silver notebooks) proceed for
# whatever did land. Tighten this to `if failed_landings:` if you'd rather
# treat any single failed table as fatal.
if len(failed_landings) == len(configs):
    raise RuntimeError("All configured datasets failed to land — aborting before bronze load.")

StatementMeta(, bcdf9bce-8279-49e7-b1e0-90803ba6c6f0, 5, Finished, Available, Finished, False)

⬇️ Downloading dataset: martinellis/nhl-game-data
Dataset downloaded to: /home/trusted-service-user/.cache/kagglehub/datasets/martinellis/nhl-game-data/versions/2
✅ Saved game_plays_players.csv → landing/game_plays_players.csv (for table 'game_plays_players')
✅ Saved game_goalie_stats.csv → landing/game_goalie_stats.csv (for table 'game_goalie_stats')
✅ Saved game_skater_stats.csv → landing/game_skater_stats.csv (for table 'game_skater_stats')
✅ Saved game_teams_stats.csv → landing/game_teams_stats.csv (for table 'game_teams_stats')
✅ Saved player_info.csv → landing/player_info.csv (for table 'player_info')
✅ Saved game_plays.csv → landing/game_plays.csv (for table 'game_plays')
✅ Saved team_info.csv → landing/team_info.csv (for table 'team_info')
✅ Saved game.csv → landing/game.csv (for table 'game')

🏁 Landing zone load complete.


 89%|████████▊ | 213M/240M [00:12<00:01, 18.9MB/s] 


### Define functions for landing into bronze

In [4]:
from pyspark.sql import functions as F, DataFrame
from pyspark.sql.utils import AnalysisException

BRONZE_SCHEMA: str = "bronze"


def load_landing_file(spark, file_path: str) -> Optional[DataFrame]:
    """Read a CSV from the landing zone.

    Returns None (with a printed warning) if the file is missing or
    unreadable, instead of letting AnalysisException crash the whole run.
    """
    try:
        df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)
    except AnalysisException as e:
        print(f"⚠️ Could not read '{file_path}': {e} — skipping.")
        return None
    return df


def add_audit_columns(df: DataFrame, source_file: str) -> DataFrame:
    """Attach bronze audit columns: ingestion timestamp and source file name."""
    return (
        df.withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.lit(source_file))
    )


def write_bronze_table(df: DataFrame, target_table: str) -> None:
    """Overwrite a bronze Delta table. Raises RuntimeError if the write fails."""
    try:
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
    except Exception as e:
        raise RuntimeError(f"Failed to write bronze table '{target_table}': {e}") from e

StatementMeta(, bcdf9bce-8279-49e7-b1e0-90803ba6c6f0, 6, Finished, Available, Finished, False)

### Run: landing → bronze

In [5]:
# Re-query in case config_datasets changed since the landing step above.
# LANDING_PATH is reused from the earlier cell in this notebook's session.
configs = get_active_configs(spark)

failed_tables: List[str] = []

for row in configs:
    table_name = row.table_name
    source_file = row.source_file

    file_path = f"{LANDING_PATH}/{source_file}"
    print(f"\n📥 Loading {source_file} → {BRONZE_SCHEMA}.{table_name}")

    df = load_landing_file(spark, file_path)
    if df is None:
        failed_tables.append(table_name)
        continue

    row_count = df.count()
    if row_count == 0:
        print(f"⚠️ '{source_file}' loaded 0 rows — skipping write for '{table_name}'.")
        failed_tables.append(table_name)
        continue

    df = add_audit_columns(df, source_file)
    target_table = f"{BRONZE_SCHEMA}.{table_name}"

    try:
        write_bronze_table(df, target_table)
    except RuntimeError as e:
        print(f"❌ {e}")
        failed_tables.append(table_name)
        continue

    print(f"✅ Wrote {target_table} (overwrite, {row_count} rows, {len(df.columns)} columns)")

if failed_tables:
    print(f"\n⚠️ {len(failed_tables)} table(s) failed to load into bronze: {failed_tables}")
    print("\n🏁 Bronze load finished with errors.")
else:
    print("\n🏁 Bronze load complete.")

# Bronze is foundational — every silver notebook depends on these tables
# existing. Fail the notebook activity on ANY table failure (not just a
# total wipeout) so Fabric marks this run Failed and downstream silver
# notebooks in the pipeline never run against incomplete bronze data.
if failed_tables:
    raise RuntimeError(f"Bronze load failed for: {failed_tables}")

StatementMeta(, bcdf9bce-8279-49e7-b1e0-90803ba6c6f0, 7, Finished, Available, Finished, False)


📥 Loading game_plays_players.csv → bronze.game_plays_players
✅ Wrote bronze.game_plays_players (overwrite, 7586604 rows, 6 columns)

📥 Loading game_goalie_stats.csv → bronze.game_goalie_stats
✅ Wrote bronze.game_goalie_stats (overwrite, 56656 rows, 21 columns)

📥 Loading game_skater_stats.csv → bronze.game_skater_stats
✅ Wrote bronze.game_skater_stats (overwrite, 945830 rows, 24 columns)

📥 Loading game_teams_stats.csv → bronze.game_teams_stats
✅ Wrote bronze.game_teams_stats (overwrite, 52610 rows, 19 columns)

📥 Loading player_info.csv → bronze.player_info
✅ Wrote bronze.player_info (overwrite, 3925 rows, 14 columns)

📥 Loading game_plays.csv → bronze.game_plays
✅ Wrote bronze.game_plays (overwrite, 5050529 rows, 20 columns)

📥 Loading team_info.csv → bronze.team_info
✅ Wrote bronze.team_info (overwrite, 33 rows, 8 columns)

📥 Loading game.csv → bronze.game
✅ Wrote bronze.game (overwrite, 26305 rows, 17 columns)

🏁 Bronze load complete.
